# RL Programming Crash Course

**PyTorch**

1. Refresher on Tensors
2. Autodiff with PyTorch
3. Supervised learning with PyTorch

**Gymnasium**

4. Intro to Gymnasium
5. Basic Q learning
6. Q learning on CartPole

(by Alex Cai)

In [ ]:
import torch
import matplotlib.pyplot as plt

## Refresher on Tensors

Similar notation to NumPy:

| Type | Shape |
| - | - |
| Scalars | `()` |
| Vectors | `(n,)` |
| Matrices | `(n, m)` |
| Tensors | `(n0, n1, ...)` |

- Keep track of what each axis indexes over
- [Named Tensor Notation](https://namedtensor.github.io/)

In [ ]:
# e.g. Dungeons and Dragons character creation
STR, DEX, CON, INT, WIS, CHA = 0, 1, 2, 3, 4, 5

# tensor with three axes:
# 5 players, 6 ability scores, 4 six-sided dice
rolls = torch.randint(low=1, high=6+1, size=(5, 6, 4))
rolls

In [ ]:
# separate axes using commas
rolls[0, :, :]  # first player's rolls

In [ ]:
rolls[:, DEX, :]  # everyone's dexterity rolls

In [ ]:
# reduce along an axis
scores = rolls.sum(dim=2)
scores

In [ ]:
# elementwise operations between matching shapes
# tensors are mutable (unlike Jax arrays)
scores -= rolls.min(dim=2).values
scores

In [ ]:
# sorting
scores[:, CON].sort(descending=True).indices

In [ ]:
# scalar operations
modifiers = (scores - 10) // 2  # modifiers
modifiers

- Now for **broadcasting**.
- Each character has a "proficiency bonus" that applies to *all* abilities.

In [ ]:
proficiency = torch.tensor([0, 2, 0, 1, 2])

# axes are right-aligned
# modifiers:   (5 players, 6 abilities)
# proficiency: (5 players,    new axis)
with_proficiency = modifiers + proficiency[:, torch.newaxis]
with_proficiency

- Now for some advanced indexing.
- Suppose the players must succeed 10 challenges.
- Each one needs one of the 6 ability scores.

In [ ]:
challenges = torch.randint(low=0, high=6, size=(10,))
challenges

In [ ]:
# index along an axis using a list or tensor
used_modifiers = with_proficiency[:, challenges]
used_modifiers

In [ ]:
# each character takes on a different challenge
characters = torch.randint(0, 5, (10,))
characters

In [ ]:
# "vectorized" indexing
with_proficiency[characters, challenges]

## Autodiff with PyTorch

- An automatic differentiation (autodiff) library and much more!
- Bread and butter for machine learning
- [Installation](https://pytorch.org/get-started/locally/) `uv pip install torch`
- Consider [Jax](https://docs.jax.dev/en/latest/) if you like functional programming. Big RL research ecosystem (Google DeepMind).
- [PyTorch Tutorials](https://docs.pytorch.org/tutorials/beginner/basics/intro.html)

Let's minimize $\text{loss}(\theta) = \theta^2$ using gradient descent. Initialize $\theta = 1$.

In [ ]:
x = torch.linspace(-1, 1, 10)
y = x ** 2
plt.xlabel(r"$\theta$"); plt.ylabel(r"loss")
plt.plot(x, y); plt.show()

In [ ]:
param = torch.tensor(1., requires_grad=True)
param

In [ ]:
loss = param ** 2
loss

In [ ]:
loss.backward()  # computes d(loss)/d(theta) at theta
param.grad

In [ ]:
# optimization algorithm
optim = torch.optim.SGD((param,), lr=0.1)
optim.step()
param

- PyTorch does *reverse-mode* autodiff (many inputs, few outputs, needs memory)
- aka backpropagation
- Complexity matches the "primal" computation
- [A great survey](https://arxiv.org/abs/1502.05767) on autodiff
- [Who Invented the Reverse Mode of Differentiation?](https://content.ems.press/assets/public/full-texts/books/251/chapters/online-pdf/978-3-98547-540-7-chapter-4949.pdf) by a pioneer
- [Jax](https://docs.jax.dev/en/latest/autodidax.html) takes a more general approach (e.g. easy higher-order derivatives and forward-mode)

## Supervised learning with PyTorch

$$
\argmin_{\theta \in \Theta}
\sum_{(x, y) \in \mathcal{D}}
\text{loss}(f_\theta(x), y)
$$

1. Function class (network architecture)
2. Data loading
3. Loss function
4. Optimization algorithm

In [ ]:
class Model(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.ff0 = torch.nn.Linear(1, 32)
        self.ff1 = torch.nn.Linear(32, 1)

    def forward(self, x):
        x = self.ff0(x)
        x = torch.nn.functional.relu(x)
        x = self.ff1(x)
        return x

In [ ]:
# create model
net = Model()
net

In [ ]:
net.ff0.weight

In [ ]:
x = torch.linspace(0, 2*torch.pi, 64)[:, torch.newaxis]
y = torch.sin(x) + 0.2 * torch.randn_like(x)
plt.scatter(x, y); plt.show()

In [ ]:
# Dataset handles data loading (e.g. from files on disk)
dataset = torch.utils.data.TensorDataset(x, y)

# DataLoader handles batching and iteration
dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True)

- PyTorch comes with many optimization algorithms.
- [AdamW](https://arxiv.org/abs/1711.05101) (Adam with decoupled weight decay) is popular.
- [Muon](https://jeremybernste.in/writing/deriving-muon) is fast for square matrices.

In [ ]:
optim = torch.optim.AdamW(net.parameters(), lr=0.1)
optim

In [ ]:
# one pass over the data (run many times)
for x_batch, y_batch in dataloader:
    optim.zero_grad()  # otherwise gradients accumulate
    pred_batch = net(x_batch)
    loss = torch.nn.functional.mse_loss(pred_batch, y_batch)
    loss.backward()  # sets `param.grad` for all parameters
    optim.step()  # updates parameters

loss

In [ ]:
# Adam tracks gradient statistics
for k, v in optim.state.items():
    print(k.shape, v.keys())

In [ ]:
plt.scatter(x, y)
plt.plot(x, net(x).detach(), color='C1'); plt.show()

In [ ]:
torch.save(net.state_dict(), 'model.pth')
! file model.pth

In [ ]:
net = Model(1, 1, 32)
net.load_state_dict(torch.load('model.pth'))

## Intro to Gymnasium

- The standard RL programming interface
- Lots of alternatives
- v0.26 made breaking `terminated vs truncated` change (see [documentation](https://farama.org/Gymnasium-Terminated-Truncated-Step-API))

In [ ]:
import gymnasium as gym

In [ ]:
# making environments
env = gym.make('CartPole-v1', render_mode='rgb_array')
env.observation_space

In [ ]:
env.action_space

In [ ]:
# sample initial state (s0 ~ mu)
# we usually don't need the second return value
obs, _info = env.reset()
obs.shape, obs.dtype, _info

In [ ]:
plt.imshow(env.render(), interpolation='none'); plt.axis('off'); plt.show()

In [ ]:
# environment step
action = 0
next_obs, reward, terminated, truncated, _info = env.step(action)
reward, terminated, truncated, _info

- In Gymnasium, the `env` object tracks state.
- Functional APIs, preferred in Jax, require you to pass state manually, but can compile end-to-end
- $s_h \sim P(\cdot \mid s_{h-1}, a_{h-1})$
- $r_h = r(s_{h-1}, a_{h-1})$ (DeepMind indexing convention)
- $\gamma_h = \gamma \cdot \mathbf{1}\{s_h \text{ is not terminal}\}$ (think about $V(s_{h-1}) = r_h + \gamma_h V(s_{h})$)

In [ ]:
env.close()

## Basic Q learning

Wrappers are helpful!

In [ ]:
from gymnasium.wrappers import NumpyToTorch

env = gym.make('CartPole-v1', max_episode_steps=100, render_mode='rgb_array')
env = NumpyToTorch(env)  # observations are already flat float32

In [ ]:
# example
network = torch.nn.Linear(env.observation_space.shape[-1], env.action_space.n)
network

In [ ]:
obs, _ = env.reset()
network(obs)

In [ ]:
optim = torch.optim.AdamW(network.parameters(), lr=0.001)

episode_length = 0
episode_lengths = []

- The following cell does one update (run many times):
- $y_{h-1} = r_h + \gamma \cdot \mathbf{1}\{ s_h \text{ is not terminal} \} \cdot \max_a Q_\theta(s_h, a)$

In [ ]:
# don't pass gradient through environment step and target
with torch.no_grad():
    # epsilon-greedy action in environment
    if torch.rand(()) < 0.9:
        action = network(obs).argmax()
    else:
        action = torch.randint(0, env.action_space.n, ())

    next_obs, reward, terminated, truncated, _info = env.step(action)
    target = reward + 0.99 * (not terminated) * network(next_obs).max()

optim.zero_grad()
pred = network(obs)[action]
loss = 0.5 * (pred - target) ** 2
loss.backward()
optim.step()

if terminated or truncated:
    episode_lengths.append(episode_length)
    episode_length = 0
    obs, _info = env.reset()
else:
    episode_length += 1
    obs = next_obs

In [ ]:
env.close()
plt.plot(episode_lengths); plt.show()

## Q learning on CartPole

- Let's use a **vectorized** environment to run many environments in parallel!
- In practice: Thousands

In [ ]:
# import wrappers from vector submodule
from gymnasium.wrappers.vector import NumpyToTorch

# make_vec instead of make
env = gym.make_vec('CartPole-v1', num_envs=8, max_episode_steps=200)
env = NumpyToTorch(env)
env.observation_space.shape

In [ ]:
env.action_space.shape

In [ ]:
obs_vec, _info = env.reset()
obs_vec

In [ ]:
# quick way to construct multi-layer perceptron
# (architecture from CleanRL)
network = torch.nn.Sequential(
    torch.nn.Linear(env.observation_space.shape[-1], 120),
    torch.nn.ReLU(),
    torch.nn.Linear(120, 84),
    torch.nn.ReLU(),
    torch.nn.Linear(84, env.single_action_space.n),
)
network(obs_vec)  # PyTorch Linear layers are "vectorized"

- See [SimBa](https://github.com/SonyResearch/simba) for notes on neural network architectures for deep RL
- [Scaling depth](https://arxiv.org/abs/2503.14858) can help
- One notable trick is ["binning" the value](https://arxiv.org/abs/2403.03950) i.e. classification instead of regression

Let's note how the implementation changes:

In [ ]:
optim = torch.optim.Adam(network.parameters(), lr=0.001)

episode_length_vec = torch.zeros(8, dtype=torch.int)
episode_lengths = []

for step in range(10_000):
    explore_vec = torch.rand((8,)) < 0.1  # usually anneal
    random_action_vec = torch.randint(0, 2, (8,))

    # compare detach() with torch.no_grad()
    exploit_action_vec = network(obs_vec).argmax(dim=-1).detach()
    action_vec = torch.where(explore_vec, random_action_vec, exploit_action_vec)
    next_obs_vec, reward_vec, terminated_vec, truncated_vec, _info_vec = env.step(action_vec)

    # Q learning targets
    target_vec = reward_vec + 0.99 * ~terminated_vec * network(next_obs_vec).detach().max(dim=-1)[0]

    # track statistics
    done_vec = terminated_vec | truncated_vec
    episode_length_vec += 1
    episode_lengths.extend(episode_length_vec[done_vec].tolist())
    episode_length_vec[done_vec] = 0

    pred_vec = network(obs_vec)[torch.arange(8), action_vec]

    # gradient update to parameters
    optim.zero_grad()
    loss = torch.nn.functional.mse_loss(pred_vec, target_vec)
    loss.backward()
    optim.step()

    obs_vec = next_obs_vec

    if (step + 1) % 1_000 == 0:
        recent_lengths = episode_lengths[-100:]
        average_episode_length = sum(recent_lengths) / len(recent_lengths)
        print(f'Step {step + 1}:\t{average_episode_length:.1f}')

In [ ]:
plt.plot(episode_lengths); plt.show()

## HUIT Open OnDemand

- https://ood.huit.harvard.edu/pun/sys/dashboard
- `My Interactive Sessions > Code Server - Universal`
- Instructions in HW3

# Wrapping up

Other helpful RL programming and debugging tips:
- [Debugging Reinforcement Learning Systems](https://andyljones.com/posts/rl-debugging.html)
- [Spinning Up in Deep RL](https://spinningup.openai.com/en/latest/)
- [Google DeepMind Jax Ecosystem](https://deepmind.google/blog/using-jax-to-accelerate-our-research/)
- [rliable](https://github.com/google-research/rliable) evaluations
- [PufferLib](https://puffer.ai/)
- HW3 for Q learning specific readings
- many more!